# NB-07-02 — PyTorch Image Classification Workbench

**Module M07 · Level 300**

Companion to lessons L07-03 and L07-05. This notebook walks the full PyTorch workflow:

1. Tensors and autograd — what a gradient actually is here.
2. `Dataset` and `DataLoader` — batching, shuffling, reproducible seeds.
3. Model construction — a dense baseline and a small CNN.
4. Training loops with validation, early stopping and curves.
5. Evaluation — confusion matrix, per-class recall, error inspection.
6. Saving and reloading a checkpoint.

The images are 8x8 synthetic digits generated by `deep_data.py`; they mirror MNIST's shape so the
pipeline transfers directly when you swap in real IDX files with `dd.load_idx_images`.

In [ ]:
import os, sys, math, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# Locate the module's code directory whether Jupyter starts here or at the repo root.
for candidate in ("../code", "code", os.path.join("modules", "M07-neural-networks-and-deep-learning", "code")):
    if os.path.isdir(candidate):
        CODE_DIR = os.path.abspath(candidate)
        break
else:
    raise RuntimeError("Run this notebook from the module's notebooks/ directory")
sys.path.insert(0, CODE_DIR)

import deep_data as dd
import nn_numpy
from nn_numpy import MLP, Adam, Momentum, RMSProp, SGD, accuracy, gradient_check, perceptron_rule, train
from torch_utils import (Trainer, SmallCNN, LSTMRegressor, TinyTransformerClassifier, AutoEncoder,
                         count_parameters, freeze, unfreeze, get_device, make_loader, set_seed,
                         load_checkpoint, save_checkpoint)
import transfer

torch.set_num_threads(2)
set_seed(0)
print("code:", CODE_DIR)
print("torch:", torch.__version__, "| device:", get_device())

## 1. Tensors and autograd

A tensor is an n-dimensional array with two extras: a device and a gradient slot. Setting
`requires_grad=True` records every operation so `backward()` can chain the derivatives.

In [ ]:
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
w = torch.tensor([[0.5], [-0.5]], requires_grad=True)
loss = ((x @ w - torch.tensor([[1.0], [1.0]])) ** 2).mean()
loss.backward()
print("loss:", round(loss.item(), 4))
print("dL/dw:\n", w.grad.numpy().round(4))

# The same derivative by hand for w[0]: dL/dw0 = 2/n * sum((x@w - 1) * x0)
pred = (x @ w).detach().numpy()
manual = 2 * np.mean((pred - 1.0) * x[:, [0]].numpy(), axis=0)
print("hand-computed dL/dw0:", manual.round(4))

Both numbers agree, which is the whole point of autograd: you write the forward pass, it writes the
chain rule.

## 2. Dataset and DataLoader

`TensorDataset` plus `DataLoader` handles shuffling and batching. The module's `make_loader` adds
seed control so every run is reproducible. When data does not fit in memory you write a `Dataset`
that reads one file per `__getitem__`; the loop below does not change at all.

In [ ]:
X, y = dd.digit_images(2400, size=8, seed=0)
print("images:", X.shape, "labels:", y.shape, "class counts:", np.bincount(y))

train_loader = make_loader(X[:1800], y[:1800], batch_size=64, seed=0)
val_loader = make_loader(X[1800:], y[1800:], batch_size=128, shuffle=False)
xb, yb = next(iter(train_loader))
print("one batch:", xb.shape, yb.shape, xb.dtype)

## 3. Two models, identical data

The MLP flattens the image and loses spatial structure. The CNN keeps the 2-D layout and shares
weights across positions, so it needs far fewer parameters per useful feature — and on images it
usually wins.

In [ ]:
results = []
for name, build in [
    ("MLP 64-32", lambda: nn.Sequential(nn.Flatten(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))),
    ("CNN w16", lambda: SmallCNN(in_channels=1, n_classes=10, width=16, image_size=8)),
]:
    set_seed(0)
    model = build()
    trainer = Trainer(model, patience=5)
    start = time.time()
    trainer.fit(train_loader, val_loader, epochs=8, lr=1e-3)
    _, acc, preds, truth = trainer.evaluate(val_loader)
    results.append((name, count_parameters(model), acc, time.time() - start, trainer, preds, truth))
    print(f"{name:10s} params {count_parameters(model):6d} val acc {acc:.3f} {time.time() - start:5.2f}s")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for name, _, _, _, trainer, _, _ in results:
    axes[0].plot(trainer.history["train_loss"], label=f"{name} train")
    axes[0].plot(trainer.history["val_loss"], "--", label=f"{name} val")
    axes[1].plot(trainer.history["val_acc"], label=name)
axes[0].set_title("loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].set_title("validation accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout(); plt.show()

## 4. Evaluation beyond one accuracy number

Overall accuracy hides class-level failures. Build the confusion matrix and read the per-class
recall — then look at the actual mistakes.

In [ ]:
name, params, acc, seconds, trainer, preds, truth = results[-1]
cm = np.zeros((10, 10), dtype=int)
for t, p in zip(truth, preds):
    cm[t, p] += 1

print("per-class recall:", (cm.diagonal() / cm.sum(axis=1)).round(3))
off = [(int(t), int(p), int(cm[t, p])) for t in range(10) for p in range(10) if t != p and cm[t, p] > 0]
print("confusions (true, predicted, count):", sorted(off, key=lambda r: -r[2])[:5])

In [ ]:
wrong = np.where(preds != truth)[0]
print("misclassified:", len(wrong), "of", len(truth))
if len(wrong):
    cols = min(6, len(wrong))
    fig, axes = plt.subplots(1, cols, figsize=(2 * cols, 2.4))
    for ax, i in zip(np.atleast_1d(axes), wrong[:cols]):
        ax.imshow(X[1800:][i, 0], cmap="gray"); ax.axis("off")
        ax.set_title(f"true {truth[i]} / pred {preds[i]}")
    plt.tight_layout(); plt.show()

## 5. Save and reload

A checkpoint holds the weights plus the configuration needed to rebuild the architecture. Loading
into an untrained model and matching accuracies proves the checkpoint is complete.

In [ ]:
path = save_checkpoint(trainer.model, "nb02_cnn.pt", config={"width": 16, "image_size": 8, "classes": 10})
set_seed(1)
reloaded = SmallCNN(in_channels=1, n_classes=10, width=16, image_size=8)
reloaded, config = load_checkpoint(reloaded, path)
_, reloaded_acc, _, _ = Trainer(reloaded).evaluate(val_loader)
print("config:", config, "| reloaded val acc:", round(reloaded_acc, 3))
os.remove(path)

## 6. Exercise

1. Train the CNN for 20 epochs and describe where validation loss starts rising.
2. Add `dropout=0.3` to the CNN and compare the train/validation gap.
3. Convert the images to 4x4 with `dd.digit_images(size=4)` and retrain. What breaks, and why?
4. Write a `torch.utils.data.Dataset` subclass reading `.npy` files lazily and plug it into
   `DataLoader`.

**References:** L07-03, L07-05. Code: `code/torch_utils.py`.